# 04 · Banco de imagens com GridFS

**Pré-requisito:** notebooks 00 a 03.

**Nesta etapa:** guardar arquivos (imagens) no MongoDB e consultá-los pelos metadados. Exercícios **5, 6 e 7**.

> **Sobre os dados:** o *FEI Face Database* (Centro Universitário FEI, São Bernardo do Campo) é liberado **apenas para fins de pesquisa**, com permissão de uso, mas **não de redistribuição ou reprodução**. Use as imagens só nesta atividade e não as publique abertamente.

In [ ]:
%pip install -q "pymongo[srv]" python-dotenv pillow numpy matplotlib requests

## Conexão com o seu banco no Atlas

A célula abaixo lê a sua URI do arquivo **`.env`** da pasta (criado no notebook 00, Etapa 6) e testa a conexão.

In [ ]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv()                                   # lê o arquivo .env da pasta
client = MongoClient(os.environ["MONGO_URI"])
client.admin.command("ping")
print("Conectado ao MongoDB Atlas")

---
## 1. Por que não guardar a imagem dentro de um documento comum?

- Um documento BSON tem limite de **16 MB**. Fotos grandes, vídeos e PDFs podem passar disso.
- Mesmo abaixo do limite, ler um documento enorme só para saber o nome do arquivo é desperdício.

## 2. GridFS: a mudança em caixas numeradas

O guarda-roupa não passa pela porta: é desmontado, cada peça vai numa caixa numerada, e uma **etiqueta-mestre** diz o que é e como remontar.

| Coleção | Conteúdo | Na analogia |
|---|---|---|
| `fs.files` | um documento por arquivo: `filename`, `length`, `uploadDate`, **`metadata`** | a etiqueta-mestre |
| `fs.chunks` | os pedaços do arquivo (até 255 KB cada), numerados `n = 0, 1, 2…` | as caixas |

`fs.files` é uma coleção **comum**: dá para filtrar, agregar e paginar pelos metadados com tudo o que você aprendeu no notebook 02.

| Comando | O que faz |
|---|---|
| `fs = gridfs.GridFS(db)` | abre o GridFS no banco |
| `fs.put(bytes, filename=..., metadata={...})` | grava e devolve o `_id` |
| `fs.get(id).read()` | lê os bytes |
| `fs.find_one({...})` / `fs.find({...})` | busca um / vários arquivos |
| `fs.exists(filename=...)` | verifica se já existe |
| `fs.delete(id)` | apaga o arquivo e todos os seus chunks |

---
## 3. Baixar o banco de faces

Vamos usar a pasta **frontalimages_manuallyaligned_part1** (cerca de 6 MB): fotos frontais, duas por pessoa.

**Convenção de nomes:** `12a.jpg` = pessoa 12, expressão **neutra** · `12b.jpg` = pessoa 12, **sorrindo**.

A célula abaixo tenta baixar e descompactar automaticamente na pasta `faces`. Se o download falhar, baixe manualmente pela página **https://fei.edu.br/~cet/facedatabase.html** (link *frontalimages_manuallyaligned_part1*), coloque o `.zip` na pasta da disciplina (a mesma deste notebook) e rode a célula de novo.

In [ ]:
import os, zipfile, glob, requests

ZIP = "frontalimages_manuallyaligned_part1.zip"
URL = "https://fei.edu.br/~cet/" + ZIP

if not os.path.exists(ZIP):
    try:
        r = requests.get(URL, timeout=60)
        r.raise_for_status()
        open(ZIP, "wb").write(r.content)
        print("Download concluído")
    except Exception as e:
        print("Não consegui baixar automaticamente:", e)
        print("Baixe manualmente em https://fei.edu.br/~cet/facedatabase.html e envie o .zip para esta pasta.")

if os.path.exists(ZIP):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("faces")

# as imagens podem estar dentro de uma subpasta: procuramos em qualquer nível
imagens = sorted(glob.glob("faces/**/*.jpg", recursive=True))
PASTA = os.path.dirname(imagens[0]) if imagens else "faces"
print(len(imagens), "imagens em", PASTA)

In [ ]:
from PIL import Image
from IPython.display import display

if imagens:
    exemplo = Image.open(imagens[0])
    print(os.path.basename(imagens[0]), exemplo.size, f"{os.path.getsize(imagens[0]) / 1024:.1f} KB")
    display(exemplo)

---
## Agora vamos fazer o Exercício 5 · A primeira imagem

**Você vai usar:** `gridfs.GridFS`, `fs.put(..., metadata=...)`, `fs.find_one`, `PIL.Image`, `io.BytesIO`.

1. Abra o GridFS no banco **`faces_db`**.
2. Grave a imagem **`1a.jpg`** com os metadados `pessoa: 1` e `expressao: "neutra"`.
3. Mostre o tamanho do arquivo (`length` em `fs.files`) e conte quantos chunks ele gerou (`fs.chunks`, filtrando por `files_id`). **Quantos chunks? Por quê?** (Lembre: cada chunk tem até 255 KB.)
4. Recupere a imagem pelo `filename` e exiba-a no notebook.
5. Abra `fs.files` e `fs.chunks` no Atlas (**Browse Collections**) e encontre o seu arquivo.

*Dica:* para exibir bytes como imagem: `Image.open(io.BytesIO(dados))`.

In [ ]:
import io, gridfs
db = client["faces_db"]
fs = gridfs.GridFS(db)
# seu código aqui

In [ ]:
# 5.2 Gravar 1a.jpg com metadados
# seu código aqui

In [ ]:
# 5.3 Tamanho e número de chunks
# seu código aqui

In [ ]:
# 5.4 Recuperar e exibir
# seu código aqui

---
## Agora vamos fazer o Exercício 6 · O banco de faces inteiro

**Você vai usar:** a lista `imagens`, `fs.exists(filename=...)`, `fs.put(..., metadata=...)`, `Image.open(...).size`.

1. Grave **todas** as imagens da pasta no GridFS.
2. **Sem duplicatas:** se o `filename` já existir, pule. Rode a célula **duas vezes**: o total não pode mudar.
3. Metadados a partir do nome do arquivo: `pessoa` (número), `expressao` (`"neutra"` para *a*, `"sorrindo"` para *b*), `largura`, `altura` e `origem: "FEI"`.
4. Ao final, mostre quantas foram **gravadas**, quantas foram **puladas** e o **total** no banco.

*Dicas:* `os.path.basename(caminho)` → `"12b.jpg"` · `"12b.jpg"[:-5]` → `"12"` · `"12b.jpg"[-5]` → `"b"`.

> A `1a.jpg` do Exercício 5 já está lá; o seu código deve pulá-la (ou atualizar os metadados dela, se quiser caprichar).

In [ ]:
# seu código aqui

### Confira o seu Exercício 6

In [ ]:
total_banco = db.fs.files.count_documents({})
duplicados = list(db.fs.files.aggregate([
    {"$group": {"_id": "$filename", "n": {"$sum": 1}}},
    {"$match": {"n": {"$gt": 1}}},
]))
sem_metadados = db.fs.files.count_documents({"metadata.pessoa": {"$exists": False}})

print("Todas as imagens no banco:", "OK" if total_banco == len(imagens) else f"FALTA ({total_banco} de {len(imagens)})")
print("Nenhum arquivo duplicado: ", "OK" if not duplicados else f"FALTA {[d['_id'] for d in duplicados][:5]}")
print("Todos com metadados:      ", "OK" if sem_metadados == 0 else f"FALTA ({sem_metadados} sem metadata.pessoa)")

---
## Agora vamos fazer o Exercício 7 · Consultando as imagens

**Você vai usar:** `aggregate` em `db.fs.files`, `$group`, `$avg`, `fs.find_one`, `matplotlib`, `fs.delete`.

1. Quantas imagens há de cada **expressão**? (agregação agrupando por `$metadata.expressao`)
2. Qual o **tamanho médio** dos arquivos, em KB? (`$avg` de `$length`)
3. **Galeria:** pessoas 1 a 5, foto neutra e sorrindo lado a lado (`plt.subplots(5, 2)`).
4. **Apague** todas as imagens de uma pessoa à sua escolha e confira o total de novo. Depois, rode o Exercício 6 outra vez para devolvê-las ao banco.

In [ ]:
# 7.1 Contagem por expressão
# seu código aqui

In [ ]:
# 7.2 Tamanho médio em KB
# seu código aqui

In [ ]:
import matplotlib.pyplot as plt
# 7.3 Galeria 5 x 2
# seu código aqui

In [ ]:
# 7.4 Apagar uma pessoa e conferir
# seu código aqui

**Próximo:** `05_App_Galeria_e_Desafio.ipynb`